# BENCHMARK 1

Comparativa de los siguientes modelos con el datasets PASCAL VOC:
- U-NET
- SegFormer
- SegNext
- SAM3 -> (Caso especial) Generalización.

In [1]:
import os
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import VOCSegmentation
import albumentations as A
import cv2
from albumentations.pytorch import ToTensorV2



### Configuración global

In [6]:
config = {
    "epochs": 50,
    "batch": 8,
    "lr": 0.0001,
    "weight_decay": 0.0001,
    "num_clases": 20,
    "img_size": 512,
    "seed": 42,
    "data": "../data/",
    "chechpoint_dir": "chechpoints/"
}

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

torch.manual_seed(config["seed"])
np.random.seed(config["seed"])

In [3]:
#Transformacion para las imágenes

transform_train = A.Compose([
    A.Resize(config["img_size"], config["img_size"],
             interpolation=cv2.INTER_LINEAR,
             mask_interpolation=cv2.INTER_NEAREST),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=mean, std=std),
    ToTensorV2()
])

transform_val = A.Compose([
    A.Resize(config["img_size"], config["img_size"],
             interpolation=cv2.INTER_LINEAR,
             mask_interpolation=cv2.INTER_NEAREST),
    A.Normalize(mean=mean, std=std),
    ToTensorV2()
])

In [8]:
class Dataset_VOC(Dataset):

    def __init__(self, root, conjunto="train", transform=None):
        self.dataset = VOCSegmentation(
            root=root,
            year="2012",
            image_set=conjunto,
            download=True
        )
        self.transform = transform
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, index:int):
        image, mask = self.dataset[index]

        image = np.array(image)
        mask = np.array(mask)

        mask[mask == 255] = 0

        if self.transform:
            augmented = self.transform(image=image,mask=mask)
            image = augmented["image"]
            mask = augmented["mask"].long()

        return image, mask

In [10]:
# Crear datasets

train_data = Dataset_VOC(config["data"], "train", transform_train)
val_data = Dataset_VOC(config["data"], "val", transform_val)

100.0%


In [ ]:
# Crear dataloaders

train_loader = DataLoader(
    train_data,
    batch_size=config["batch"],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_data,
    batch_size=config["batch"],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)